In [ ]:
import json
import os
import re
from openbabel import openbabel, pybel
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import rdBase

# ================= 1. 超参数设置 (Configuration) =================
HYPERPARAMS = {
    # Puppy 集合的限制
    "PUPPY_MAX_HEAVY_ATOMS": 7,      # 小于等于该重原子数的分子进入 Puppy (前提：无金属，无*，无自由基，MMFF94支持)

    # Filter 集合的限制 (互斥逻辑)
    "FILTER_MIN_RINGS": 3,               # 环数量 >= 该值 -> Filter
    "FILTER_MIN_HETERO_CHAIN": 5,        # 无环 且 杂原子数 >= 该值 -> Filter
    "FILTER_MIN_CARBON_CHAIN": 7,        # 无环 且 碳原子数 >= 该值 -> Filter
    
    # 环数量 < 该值 但拥有的碳原子数量 >= 该值 -> Filter
    "FILTER_MAX_CARBON_WITH_RINGS": 14,  
    # 包含的非常见同位素(如[2H])数量 >= 该值 -> Filter
    "FILTER_MIN_UNCOMMON_ISOTOPES": 2,
}

# ================= 2. 路径配置 =================
base_dir = "processed_data" 
input_path = os.path.join(base_dir, "fragment_vocabulary_64_dynamic_margin.json")

# 输出文件路径
output_filter_path = os.path.join(base_dir, "fragment_vocabulary_filter.json")
output_frame_path = os.path.join(base_dir, "fragment_vocabulary_frame.json")
output_linker_path = os.path.join(base_dir, "fragment_vocabulary_linker.json")
output_puppy_path = os.path.join(base_dir, "fragment_vocabulary_puppy.json")
output_valid_path = os.path.join(base_dir, "fragment_vocabulary_valid.json") 

# ================= 3. 辅助函数 =================
def save_compact_json(data, filepath):
    """
    保存JSON，保持整体缩进，但将向量列表压缩在一行显示，方便人类阅读。
    """
    if not data: 
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("{}")
        return

    json_str = json.dumps(data, indent=4)
    # 正则匹配：[ 换行/空格 数字... ] -> 压缩为 [ 数字... ]
    pattern = r'\[\s*([-\d\.\,\seE]+?)\s*\]'
    
    def compact_list(match):
        content = match.group(1)
        content = " ".join(content.split())
        content = content.replace(" ,", ",").replace(", ", ",").replace(",", ", ")
        return f"[{content}]"

    json_str = re.sub(pattern, compact_list, json_str)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(json_str)
    print(f"Saved {len(data)} entries to {os.path.basename(filepath)}")


# ================= 4. 核心处理逻辑 =================
def process_vocabulary_classification(input_file, config):
    # 0. 屏蔽底层库的满屏警告
    openbabel.obErrorLog.SetOutputLevel(0) 
    rdBase.DisableLog('rdApp.warning')
    rdBase.DisableLog('rdApp.error')

    # 1. 检查输入文件
    if not os.path.exists(input_file):
        print(f"Error: Input file not found: {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        vocab = json.load(f)
    
    # 2. 初始化分类字典
    dict_filter = {} 
    dict_frame = {}  
    dict_linker = {} 
    dict_puppy = {}  
    dict_valid = {}  
    
    # 统计数据
    stats = {
        "total": len(vocab),
        "error": 0,
        "special_token": 0,
        "wildcard": 0,        
        "mmff94_reject": 0,     # RDKit 动态力场拒绝统计
        "metal": 0,
        "radical": 0,
        "isotope_filter": 0,    
        "rings_filter": 0,      
        "rings_c_filter": 0,    
        "chain_c_filter": 0,    
        "chain_het_filter": 0   
    }
    
    print(f"Start processing {len(vocab)} entries...")
    print(f"Using Config: {json.dumps(config, indent=2)}")
    
    for smiles, embedding in vocab.items():
        
        # --- A. 特殊 Token 处理 ---
        if smiles.startswith("<") and smiles.endswith(">"):
            # 特殊占位符默认放入所有集合备用
            dict_filter[smiles] = embedding[:]
            dict_frame[smiles] = embedding[:]
            dict_linker[smiles] = embedding[:]
            dict_puppy[smiles] = embedding[:]
            dict_valid[smiles] = embedding[:]
            stats["special_token"] += 1
            continue 

        # --- B. 通配符 * 检测 ---
        if "*" in smiles:
            dict_filter[smiles] = embedding
            stats["wildcard"] += 1
            continue 

        # --- C. RDKit 动态 MMFF94 嗅探 ---
        try:
            rd_mol = Chem.MolFromSmiles(smiles, sanitize=True)
            if rd_mol is None:
                raise ValueError("RDKit parse failed")
                
            # 去除坐标氢/显式氢，确保力场引擎只评估重原子骨架和隐式氢
            rd_mol = Chem.RemoveHs(rd_mol)
            
            # 动态检测：如果无法分配力场参数(如含B原子)，直接过滤
            if not AllChem.MMFFHasAllMoleculeParams(rd_mol):
                dict_filter[smiles] = embedding
                stats["mmff94_reject"] += 1
                continue
                
        except Exception:
            # 任何 RDKit 解析导致的底层错误，均视为不安全分子，丢入垃圾桶
            dict_filter[smiles] = embedding
            stats["error"] += 1
            continue
            
        # --- D. OpenBabel 拓扑结构与规则解析 ---
        try:
            mol = pybel.readstring("smi", smiles)
            if not mol or len(mol.atoms) == 0:
                raise ValueError("OpenBabel returned empty molecule")
            
            # 不完整成键/自由基检测
            is_radical = any([atom.spin > 1 for atom in mol.atoms])
            if is_radical:
                dict_filter[smiles] = embedding
                stats["radical"] += 1
                continue

            # 获取化学属性
            heavy_atoms = [atom for atom in mol.atoms if atom.atomicnum != 1]
            num_heavy = len(heavy_atoms)
            num_rings = len(mol.sssr)
            num_carbon = len([a for a in heavy_atoms if a.atomicnum == 6])
            num_hetero = len([a for a in heavy_atoms if a.atomicnum != 6])
            has_metal = any([a.OBAtom.IsMetal() for a in mol.atoms])
            
            # 同位素检测 (自然丰度同位素 isotope 值为 0)
            num_isotopes = sum(1 for atom in mol.atoms if atom.isotope != 0)

            # ================= 逻辑 1: Puppy =================
            if num_heavy <= config["PUPPY_MAX_HEAVY_ATOMS"] and not has_metal:
                dict_puppy[smiles] = embedding

            # ================= 逻辑 2: Filter (几何与组成过滤) =================
            is_filtered = False
            
            # 1. 金属
            if has_metal:
                dict_filter[smiles] = embedding
                stats["metal"] += 1
                is_filtered = True
            
            # 2. 同位素过多
            elif num_isotopes >= config["FILTER_MIN_UNCOMMON_ISOTOPES"]:
                dict_filter[smiles] = embedding
                stats["isotope_filter"] += 1
                is_filtered = True
            
            # 3. 环太多
            elif num_rings >= config["FILTER_MIN_RINGS"]:
                dict_filter[smiles] = embedding
                stats["rings_filter"] += 1
                is_filtered = True

            # 4. 有环 但 碳原子数超标
            elif num_rings > 0 and num_carbon >= config["FILTER_MAX_CARBON_WITH_RINGS"]:
                dict_filter[smiles] = embedding
                stats["rings_c_filter"] += 1
                is_filtered = True
                
            # 5. 无环 但 碳链太长
            elif num_rings == 0 and num_carbon >= config["FILTER_MIN_CARBON_CHAIN"]:
                dict_filter[smiles] = embedding
                stats["chain_c_filter"] += 1
                is_filtered = True
                
            # 6. 无环 但 杂原子链太长
            elif num_rings == 0 and num_hetero >= config["FILTER_MIN_HETERO_CHAIN"]:
                dict_filter[smiles] = embedding
                stats["chain_het_filter"] += 1
                is_filtered = True
            
            if is_filtered:
                continue
                
            # ================= 逻辑 3: Valid / Frame / Linker =================
            dict_valid[smiles] = embedding
            
            if num_rings > 0:
                dict_frame[smiles] = embedding
            else:
                dict_linker[smiles] = embedding

        except Exception:
            # 捕获任何 OpenBabel 解析或处理时的异常
            dict_filter[smiles] = embedding
            stats["error"] += 1

    # 3. 打印统计信息
    print("\n" + "=" * 60)
    print("Classification Statistics:")
    print(f"Total Input      : {stats['total']}")
    print(f"Special Tokens   : {stats['special_token']}")
    print("-" * 60)
    print(f"[Filter] Total   : {len(dict_filter)}")
    print(f"   - MMFF94 Rejected (e.g. B, As) : {stats['mmff94_reject']}  <-- [Dynamic Forcefield Check]")
    print(f"   - Contains '*'                 : {stats['wildcard']}")
    print(f"   - Radicals (Incomplete)        : {stats['radical']}")
    print(f"   - Metals                       : {stats['metal']}")
    print(f"   - Isotopes >={config['FILTER_MIN_UNCOMMON_ISOTOPES']:<2}               : {stats['isotope_filter']}")
    print(f"   - Rings >={config['FILTER_MIN_RINGS']:<2}                  : {stats['rings_filter']}")
    print(f"   - Rings <{config['FILTER_MIN_RINGS']} BUT C >={config['FILTER_MAX_CARBON_WITH_RINGS']:<2}        : {stats['rings_c_filter']}")
    print(f"   - No Ring BUT C >={config['FILTER_MIN_CARBON_CHAIN']:<2}         : {stats['chain_c_filter']}")
    print(f"   - No Ring BUT Het >={config['FILTER_MIN_HETERO_CHAIN']:<2}       : {stats['chain_het_filter']}")
    print(f"   - RDKit/OB Exceptions          : {stats['error']}")
    print("-" * 60)
    print(f"[Valid] Total    : {len(dict_valid)}  (Excluding Filter)")
    print(f"   |- [Frame]    : {len(dict_frame)}")
    print(f"   |- [Linker]   : {len(dict_linker)}")
    print("-" * 60)
    print(f"[Puppy] Total    : {len(dict_puppy)}")
    print("=" * 60 + "\n")

    # 4. 保存结果
    save_compact_json(dict_filter, output_filter_path)
    save_compact_json(dict_frame, output_frame_path)
    save_compact_json(dict_linker, output_linker_path)
    save_compact_json(dict_puppy, output_puppy_path)
    save_compact_json(dict_valid, output_valid_path)


# ================= 5. 执行入口 =================
if __name__ == "__main__":
    if not os.path.exists(base_dir):
        print(f"Directory '{base_dir}' does not exist. Attempting to create it...")
        os.makedirs(base_dir, exist_ok=True)
        
    process_vocabulary_classification(input_path, HYPERPARAMS)